# Burfisher 3e: TBL94

Replicates exercise TBL94 of Burfisher, *Introduction to Computable General Equilibrium Models* (3rd ed.), from scratch, on the book's nus333 database (2 regions × 3 sectors), and every level is checked against GEMPACK's solution of the same `.EXP`: the notebook fails unless every cell is within 0.5%.

Run it from anywhere inside the repository.

In [ ]:
import sys
from pathlib import Path

# the repository root, wherever the notebook is opened from
ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents) if (d / "pyproject.toml").exists())
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts" / "gtap")]

from pyomo.environ import value

from equilibria._local_refs import nus333_dir, path_capi_src
from equilibria.blocks.gtap import ClosureBlock, ShockBlock, overwrite, ppriv_tornqvist
from equilibria.blocks.gtap._derived_params import xscale_data
from equilibria.templates.gtap import GTAPParameters
from equilibria.templates.gtap.gtap_block_model import build_block_model
from equilibria.templates.gtap.gtap_contract import GTAPClosureConfig
from equilibria.templates.gtap.gtap_multiperiod_driver import solve_multiperiod
from equilibria.templates.gtap.shocks import apply_shock

sys.path.insert(0, str(path_capi_src()))  # the PATH solver's Python bindings
# hooks from an earlier exercise in this kernel would leak into this one (#214)
overwrite.clear()

## 1. Data: nus333 and the exercise's parameters (`default.prm`)

In [ ]:
har = nus333_dir()
p = GTAPParameters()
p.load_from_har(
    basedata_path=har / "basedata.har",
    sets_path=har / "sets.har",
    default_path=har / "default.prm",
    baserate_path=har / "baserate.har",
)

## 2. The exercise's closure, with `@overwrite`

A regulation cuts the output of MFG by the MFG activity of USA by 1%. The model finds the production tax that has the same effect: the shadow tax of the regulation. GEMPACK's `TBL94.EXP` swaps `qca("MFG","MFG","USA")` for `to("MFG","MFG","USA")` and shocks `qca = -1`. Here:

- the production tax `prdtx_rai[USA,MFG,MFG]` becomes endogenous;
- `b.target` declares the target `qca_target` on output `x`: a new variable, fixed at 1, and the row `x[shock] = qca_target × x[check]`.

One unknown freed, one row added: the system stays square.

In [ ]:
CELL = ("USA", "MFG", "MFG")


@overwrite(ShockBlock, period="shock")
def regulation(b):
    b.endogenous("prdtx_rai", CELL)
    b.target(
        "qca_target",
        CELL,
        quantity=lambda m, r, a, i: m.x[r, a, i],
        domains=("r", "a", "i"),
    )

## 3. GTAP closure and model (base / check / shock)

In [ ]:
gc = GTAPClosureConfig(
    name="base",
    closure_type="MCP",
    capital_mobility="sluggish",
    fix_endowments=False,
    fix_taxes=False,
    fix_technology=False,
    if_sub=False,
    savf_flag="capFlex",
    numeraire="pnum",
)
# Like GAMS compStat, the base is not solved: its levels are the reference of the
# Fisher indices of check and shock.
m, _ = build_block_model(p, p.sets, gc, "ROW", base_calibrated=False, ref_gdx=None)

## 4. The shock

As in the `.EXP`, in % change; `apply_shock` converts each instrument to levels and writes it in the shock period only.

In [ ]:
SHOCK = {
    "qca_target": {("USA", "MFG", "MFG"): -1.0},
}
apply_shock(m, SHOCK)

## 5. Solve check and shock

In [ ]:
res = solve_multiperiod(
    m,
    p,
    gc,
    ref_gdx=None,
    skip_base_solve=True,
    mute_welfare=True,
    seed_from_prior=False,
    mode="gtap",
    solve_check=True,
    # the shock in 4 steps, each solve starting from the previous one
    shock_steps=(0.25, 0.5, 0.75, 1.0),
)
codes = {k: int(res[k]["code"]) for k in res}
print("codes:", codes)
assert set(codes.values()) == {1}, codes

## 6. Every level against GEMPACK

In [ ]:
import compare_burfisher_gempack as cbg
import compare_gempack_levels as cg

EXP = "TBL94"
GRAGG = "gragg"
report = cbg.run(EXP, cg.model_levels(m), dict(xscale_data(p, p.sets)), GRAGG)
o = report.overall
w = o["pct_within"]
print(f"{o['cells']} cells (pre and post) against GEMPACK {cbg.method(EXP, GRAGG)}")
print(f"within 0.1%: {w[1e-3]:.2f}%   within 0.5%: {w[5e-3]:.2f}%   within 1%: {w[1e-2]:.2f}%")
print("\nlargest differences:")
print(f"{'variable':10s}{'period':>7s}  {'cell':24s}{'equilibria':>14s}{'GEMPACK':>14s}{'diff':>9s}")
for c in sorted(report.cells, key=lambda c: -c.d)[:5]:
    cell = ",".join(c.index)
    print(f"{c.variable:10s}{c.period:>7s}  {cell:24s}{c.a:14.6g}{c.b:14.6g}{100 * c.d:8.3f}%")

assert o["max"] <= 5e-3, f"{EXP}: not every cell within 0.5% of GEMPACK"
print("every cell within 0.5% of GEMPACK")